# Rama Bench Physics-Loss Fine-Tuning

**This notebook sends real commands to Rama's DM and reads real camera frames over `dao` shared memory.** -- `dao` and the bench's shared-memory segments only exist on the bench control computer. Everything in this notebook that does *not* touch real hardware (instrument construction, the OPD-to-actuator projection, the training-loop bookkeeping, the loss) has been smoke-tested here in **`DRY_RUN`** mode (see below), which substitutes the real DM/camera calls with the existing synthetic `wfs`/`dm` forward model so the surrounding logic can be checked without hardware.

**Before running with `DRY_RUN = False` on the real bench, confirm:**
1. The SHM paths in the "Bench I/O" section still match the current setup (copied from `prototyping/misreg_test.ipynb`).
2. The crop/tile geometry constants still match what `Tutorials/Rama/CalibrateExampleRamaTwin.ipynb` used to calibrate `RamaWFS.pth` -- if the camera or pyramid alignment has changed since that calibration, these are stale.

## Relationship to `Ideas/08-unsupervised-physics-finetuning.md`

That plan fine-tunes a reconstructor using only `Physics_loss` -- a bare reprojection-consistency loss that never needs the true OPD, because `Physics_loss.compute()`'s return value algebraically reduces to a function of the network's own prediction and the *observed* WFS frame, with the ground-truth-derived terms cancelling out (see that doc's step 4 for the full derivation). There, "observed frame" meant a frame generated by the *simulator's own* `wfs.forward()`, which made the whole fine-tune reusable through `Trainer.train()` unmodified.

Here, "observed frame" means a frame from the **real Rama camera**, produced by physically realizing each `PhaseDataset` OPD sample on the real DM. This is a genuinely more faithful version of the same idea: the captured frame now carries real, un-modeled optical/detector physics (whatever `RamaWFS.pth`'s calibration didn't fully capture), not just simulator noise. But `Trainer.train()` can't run this directly -- its body always calls `self.wfs(residual_opd, pupilGT)` in one vectorized, pure-Python/torch line (`AI4AO/Trainer.py:101`) to produce the frame, and there's no way to swap in real hardware I/O for that one line without hand-rolling the loop. This notebook does exactly that: **it takes `Trainer.train()`'s body and replaces that one line with a per-sample real-hardware for-loop**, keeping everything else (the closed-loop bookkeeping, the reconstruction, the backward pass) the same.

One honest caveat on the "unsupervised" framing: the wavefronts shown to the real WFS here are still *synthetic* (drawn from `PhaseDataset`, physically realized via the DM), not real atmospheric turbulence -- this fine-tunes the network against the real WFS/camera's own un-modeled response to *known* wavefronts, the same way a real interaction-matrix calibration pokes known shapes on the DM to calibrate against the bench's real response. It is not yet fine-tuning against real, unknown atmospheric data. The reconstructor itself still never sees the true OPD, only the captured frame -- so the core "nothing ground-truth-derived reaches the optimizer" property from `Ideas/08` still holds, just with "physically realized" now meaning "realized via the real DM and camera" rather than "realized via the simulator."

In [ ]:
import time
import copy

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from mmengine import Config
from tqdm import tqdm

from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, TwinCalibrator, imshow_multiple, imshow
from AI4AO.LossFunctions import Physics_loss
from AI4AO.paths import DATA_DIR

device = 'cuda'  # set to "cpu" if CUDA is not available

# DRY_RUN=True replaces every real-hardware call (DM_SHM, WFS_CAM_SHM, the 20 ms wait) with
# the existing synthetic wfs/dm forward model, so the rest of this notebook's logic can be
# smoke-tested on a machine without `dao`/the bench. Set to False only on the bench itself,
# and only after working through the "Before running on the real bench" checklist above.
DRY_RUN = True

# ON_SKY selects whether to send commands to the DM or just use the atmospheric turbulence
# to generate the dataset
ON_SKY = False

if not DRY_RUN:
    import dao

## Loading the Rama instrument

Same params file, same calibrated twin, same elongated-mask step and modes-to-commands matrix as `Tutorials/Rama/TrainExampleRama.ipynb` -- this notebook fine-tunes the *same* reconstructor that notebook trains, it just sources training frames from the real bench instead of the simulator for the fine-tuning phase below.

In [ ]:
paramfile = 'Rama_params.py'
PATH = DATA_DIR / "Rama"

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']

WFSParams["centralObstruction"] = 0.0

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

wfs = PyramidWFS(WFSParams, device)
wfs.LoadCalibration(PATH / "RamaWFS.pth")
wfs.eval()

dm = DeformableMirror(WFSParams, DMParams, device)
dm.LoadCalibration(PATH / "RamaDM.pth")
dm.eval()
# with torch.no_grad():
#     dm.rotationAngle = dm.rotationAngle + 20.
#     dm.grid_shift = dm.grid_shift + 1



framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

M2C = np.load(PATH / r"M2C.npy").astype(np.float32)
M2C = torch.from_numpy(M2C).to(device=device, dtype=torch.float32)
M2C = M2C[:, :DMParams["Nmodes"]]
M2C_T = M2C.T

nacts = int(dm.totalAct)
print(f"Actuator count (dm.totalAct): {nacts}")

## Reconstructor -- resume from the existing trained checkpoint

`PWFSNet`, copied verbatim from `TrainExampleRama.ipynb` (the deeper Rama-specific variant, two conv layers per resolution stage). This is the network `TrainExampleRama.ipynb` already trained against the simulator; this notebook resumes from that same checkpoint (`RamaCNN.pth`) as its starting point, then fine-tunes it further against real bench frames. As in `Ideas/08`'s Phase A/Phase B split, this fine-tune needs a reasonable starting point, not a from-scratch network -- that starting point is `RamaCNN.pth`, already produced by ordinary supervised training.

In [ ]:
class PWFSNet(nn.Module):
    def __init__(self, DMParams):
        super().__init__()

        Nmodes = DMParams["Nmodes"]

        self.stem = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=11, padding=5, groups=4),
            nn.GELU(),

            nn.Conv2d(32, 64, kernel_size=7, padding=3, groups=4),
            nn.GELU(),

            nn.MaxPool2d(2),
        )

        self.encoder = nn.Sequential(
            nn.Conv2d(64, 64, 5, padding=2),
            nn.GELU(),
            nn.Conv2d(64, 64, 5, padding=2),
            nn.GELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.GELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.GELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, 2, padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, Nmodes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.encoder(x)
        return self.head(x)


phaseReconstructor = PWFSNet(DMParams).to(device=device)

total_params = sum(p.numel() for p in phaseReconstructor.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

LOAD_CHECKPOINT_PATH = PATH / "RamaCNN.pth"
SAVE_CHECKPOINT_PATH = PATH / "RamaCNN.pth"

# Built to reuse Trainer's save_checkpoint/load_checkpoint (and its z_inv) for free;
# trainer.train() is deliberately never called -- this notebook hand-rolls that loop below
# so it can source frames from the bench instead.
optimizer_bench = torch.optim.AdamW(phaseReconstructor.parameters(), lr=3e-5, fused=True)
trainer = Trainer(
    wfs=wfs,
    framePreprocessor=framePreprocessor,
    dm=dm,
    M2C=M2C,
    phaseReconstructor=phaseReconstructor,
    dataset=dataset,
    loss=Physics_loss(wfs=wfs),
    optimizer=optimizer_bench,
)

try:
    trainer.load_checkpoint(LOAD_CHECKPOINT_PATH, load_optimizer=False)
except KeyError:
    # A checkpoint saved by the older torch.save(state_dict, ...) pattern is a raw state
    # dict, not the wrapped format save_checkpoint/load_checkpoint expects -- see
    # TrainExampleRama.ipynb's identical note.
    print("Starting from scratch")

## DM command projection -- "project the OPD into influence functions"

`dm.forward(coefs)` computes `einsum('rc,cwh->rwh', coefs, self.IF)`: exactly linear in `coefs`. Feeding it the identity matrix returns `self.IF` itself (one influence function per actuator), so the least-squares zonal command that best reproduces a target OPD through the DM's own (calibrated) influence functions is a single pseudo-inverse, computed once since `dm` is frozen `.eval()` and `self.IF` never changes in this notebook:

```
IF_flat = dm(eye(nacts)).flatten(-2)      # (nacts, Npix)
IF_pinv = pinv(IF_flat)                    # (Npix, nacts)
command  = opd.flatten(-2) @ IF_pinv       # (nacts,)
```

**`DM_COMMAND_SCALE` below is an unconfirmed placeholder, not a calibrated value.** `RamaDM.pth` was fit (via `TwinCalibrator.rough_calibrate_dm`/`fit_dm_and_offsets`) against a real bench interaction matrix built from real `DM_SHM.set_data(coefs)` pokes, which is a good reason to expect `dm.forward`'s `coefs` convention already matches `DM_SHM`'s own units directly (calibration would have absorbed a fixed unknown scale into `dm.sign`). But `prototyping/misreg_test.ipynb` sends a related quantity to `DM_SHM` as `-dmFlat / 10` in one place, which could mean a real scale/sign mismatch exists, or could just be a deliberate cautious fraction of a computed correction and an aberration-cancelling sign flip -- the evidence available doesn't settle which. **Do not trust `DM_COMMAND_SCALE = 1.0` by default; run the single-poke sanity check near the end of this notebook on the real bench first**, and adjust this constant (and/or its sign) based on what you observe before running the full fine-tune.

In [ ]:
zonal_basis = torch.eye(nacts, device=device, dtype=torch.float32)
IF_flat = dm(zonal_basis).flatten(start_dim=-2)   # (nacts, Npix) -- dm.eval() keeps self.IF fixed
IF_pinv = torch.linalg.pinv(IF_flat)               # (Npix, nacts)

STROKE_LIMIT = 1  # confirm the real value, then uncomment the clamp in opd_to_bench_command

def opd_to_bench_command(opd):
    """Least-squares zonal actuator command (dm.forward's own `coefs` convention) that
    reproduces `opd` (Nres, Nres) through the DM's calibrated influence functions."""
    command = opd.flatten(start_dim=-2) @ IF_pinv
    command = torch.clamp(command, -STROKE_LIMIT, STROKE_LIMIT)
    return command

## Bench I/O

SHM paths and the crop/tile geometry are reused verbatim from `prototyping/misreg_test.ipynb` and `Tutorials/Rama/CalibrateExampleRamaTwin.ipynb` respectively -- **not re-derived here**, since guessing new values for either would silently desync this notebook from whatever `RamaWFS.pth` was actually calibrated against. `_COORDS`/`PUPIL_SIZE`/`PUPIL_SEPARATION`/`FULL_SHAPE` in particular must match `CalibrateExampleRamaTwin.ipynb`'s values exactly, or the real frame's pupils will land in different pixels than `wfs.pupil_centers`/`wfs.reference_intensity` were calibrated against -- with no error raised, just a silently misaligned fine-tune.

Real frames go through `TwinCalibrator.tile_pyramid_frame` (the same crop/stitch method used to build `iMat_bench` at calibration time), **not** `misreg_test.ipynb`'s own `GetROI` helper -- that helper uses a different, simpler corner size (`n=60`) that doesn't match the calibration's actual `pupil_size + 2*pupil_separation = 48`, so reusing it here would silently crop/tile the live frame differently than `RamaWFS.pth`'s own calibration did.

In [ ]:
BENCH_SETTLE_TIME_S = 10e-3  # wait for the DM to settle before the camera integrates


if not DRY_RUN:
    WFS_CAM_SHM_PATH = "/tmp/pyrIm.im.shm"
    WFS_CAM_BACKGROUND_SHM_PATH = "/tmp/pyrBg.im.shm"
    WFS_CAM_EXTRACT_SHM_PATH = "/tmp/pyrImCalExtract.im.shm"
    DM_SHM_PATH = '/tmp/dm1Cmd05.im.shm'
    MODAL_VECTOR_INFERENCE_SHM_PATH = "/tmp/pyrModesNN.im.shm"
    ZONAL_VECTOR_INFERENCE_SHM_PATH = "/tmp/dm1ResWfNN.im.shm"
    VALID_PIXEL_SHM_PATH = "/tmp/pyrIllum.im.shm"
    PYR_PUP_SHM_PATH = "/tmp/pyrPup.im.shm"


    WFS_CAM_SHM = dao.shm(WFS_CAM_SHM_PATH)
    WFS_CAM_BACKGROUND = dao.shm(WFS_CAM_BACKGROUND_SHM_PATH)
    WFS_CAM_EXTRACT = dao.shm(WFS_CAM_EXTRACT_SHM_PATH)
    DM_SHM = dao.shm(DM_SHM_PATH)
    MODAL_VECTOR_INFERENCE_SHM = dao.shm(MODAL_VECTOR_INFERENCE_SHM_PATH)
    ZONAL_VECTOR_INFERENCE_SHM = dao.shm(ZONAL_VECTOR_INFERENCE_SHM_PATH)
    valid_pixels = dao.shm(VALID_PIXEL_SHM_PATH).get_data()  != 0
    pyrPup = dao.shm(PYR_PUP_SHM_PATH).get_data()  != 0

    assert nacts == DM_SHM.get_data().shape[0], (
        "DM_SHM's command length doesn't match dm.totalAct -- the loaded RamaDM.pth "
        "geometry and the live DM_SHM handle disagree on actuator count."
    )


    def GetROI(img, n = 60):
        """
        Extract the four n x n corners of an image and stitch them together.

        Parameters
        ----------
        img : np.ndarray
            Input 2D image.
        n : int
            Size of each corner.

        Returns
        -------
        stitched : np.ndarray
            Image of shape (2*n, 2*n):
                top-left     | top-right
                -------------+-------------
                bottom-left  | bottom-right
        """
        if img.ndim != 2:
            raise ValueError("Input image must be 2D.")

        h, w = img.shape
        if n > h or n > w:
            raise ValueError("Corner size is larger than the image.")

        tl = img[:n, :n]
        tr = img[:n, -n:]
        bl = img[-n:, :n]
        br = img[-n:, -n:]

        top = np.hstack((tl, tr))
        bottom = np.hstack((bl, br))

        return np.vstack((top, bottom))

    extract = GetROI(pyrPup)
    valid_pixels_extract = GetROI(valid_pixels)
    FRAME = np.zeros((120,120), dtype=np.float32)


    def GetFrame():

        # frame = WFS_CAM_SHM.get_data(check=True, semNb=4).astype(np.float32) - WFS_CAM_BACKGROUND.get_data()
        FRAME[extract] = WFS_CAM_EXTRACT.get_data(check=True, semNb=4).squeeze()
        frame = FRAME# * valid_pixels_extract
        norm = frame[valid_pixels_extract].sum()
        tiled = GetROI(frame, 60)
        if norm > 0:
            return frame / norm
        else:
            frame = WFS_CAM_SHM.get_data(check=True, semNb=4).astype(np.float32) - WFS_CAM_BACKGROUND.get_data()
            norm = frame[valid_pixels].sum()
            tiled = GetROI(frame, 60)
            return tiled / norm

    def GetFrameSlice(n=60):
        # Top-left
        tl = WFS_CAM_SHM.get_data(
            check=True, semNb=4,
            x=slice(0, n),
            y=slice(0, n)
        )

        # Top-right
        tr = WFS_CAM_SHM.get_data(
            x=slice(-n, None),
            y=slice(0, n)
        )

        # Bottom-left
        bl = WFS_CAM_SHM.get_data(
            x=slice(0, n),
            y=slice(-n, None)
        )

        # Bottom-right
        br = WFS_CAM_SHM.get_data(
            x=slice(-n, None),
            y=slice(-n, None)
        )

        top = np.hstack((tl, tr))
        bottom = np.hstack((bl, br))

        return np.vstack((top, bottom)).astype(np.float32)

    def GetBenchFrame():
        """One real, background-subtracted, calibration-consistent-tiled camera frame."""
        return torch.from_numpy(GetFrame()).to(device=device, dtype=torch.float32)


def GetBenchWFSFrames(residual_opd):
    """Physically realize each OPD sample in `residual_opd` (batch, Nres, Nres) on the DM
    and capture the resulting WFS frame, one sample at a time (a DM can only show one shape
    at once) -- the hardware replacement for `self.wfs(residual_opd, pupilGT)` in
    Trainer.train() (AI4AO/Trainer.py:101)."""

    if DRY_RUN:
        # Simulate a bench frame
        with torch.no_grad():
            frame = wfs(residual_opd)
        return frame

    frames = []
    last_command_np = None
    for i in range(residual_opd.shape[0]):
        if not ON_SKY:
            command = opd_to_bench_command(residual_opd[i])
            command_np = command.detach().cpu().numpy().astype(np.float32).reshape(-1, 1)
            last_command_np = command_np
            DM_SHM.set_data(command_np)
            time.sleep(BENCH_SETTLE_TIME_S)
        frames.append(GetBenchFrame())

    if last_command_np is not None:
        DM_SHM.set_data(np.zeros_like(last_command_np))  # flatten the DM once the batch is done

    return torch.stack(frames)

## The hand-rolled bench fine-tuning loop

This mirrors `Trainer.train()`'s body (`AI4AO/Trainer.py:53-139`) closely: the outer per-step atmosphere draw, the leaky-integrator correction bookkeeping, and the backward/optimizer step are all unchanged in *shape*. The one substantive change is the line that produces `wfs_frames`: instead of `self.wfs(residual_opd, pupilGT)` (one vectorized synthetic call), it's `GetBenchWFSFrames(residual_opd)` (a per-sample real-hardware for-loop, or its `DRY_RUN` stand-in). `wfs`/`dm` stay `.eval()` throughout -- only `phaseReconstructor` trains, and neither `wfs.train()` nor `dm.train()` should ever be called in this notebook.

Kept intentionally simple relative to `Ideas/08`'s synthetic demo: no `loss_tracker_ideal` oracle diagnostic (it would need its own bench-realized "ideal" frame, doubling the real hardware cost per step for a number that isn't essential to actually doing the fine-tune), and `CLOSED_LOOP_ITERATIONS` defaults to `1` (open-loop, single-frame realization per sample) -- each `BENCH_FINETUNE_STEPS` step already costs `Nphases * (BENCH_SETTLE_TIME_S + camera integration/readout)` of real wall-clock time on the bench; going beyond `closed_loop_iterations = 1` multiplies that further and has not been considered here for how the leaky integrator's state should interact with the bench delay. Start `BENCH_FINETUNE_STEPS` small (a handful) to confirm the whole pipeline runs correctly on the real bench before committing to a long run.

In [ ]:
ref_opd = torch.nn.Parameter(torch.zeros(*wfs.pupil.shape, device=device, dtype=torch.float32))
ref_pupil = torch.nn.Parameter(torch.ones(*wfs.pupil.shape, device=device, dtype=torch.float32))

optimizer_bench = torch.optim.AdamW([
                                    {"params": phaseReconstructor.parameters(), "lr": 3e-5},
                                    {"params": ref_opd, "lr": 3e-3},
                                    {"params": ref_pupil, "lr": 3e-3},
                                    #{"params": [dm._rotationAngle, dm._grid_shift], "lr": 3e-2}
                                    ],
                                    fused=True
                                    )



In [ ]:
BENCH_FINETUNE_STEPS = 100  # start small on the real bench -- see markdown above
CLOSED_LOOP_ITERATIONS = 1

loss_tracker_bench = torch.zeros(BENCH_FINETUNE_STEPS, device=device)

phaseReconstructor.train()

progressBar = tqdm(range(BENCH_FINETUNE_STEPS))
for u in progressBar:
    with torch.no_grad():
        batch = dataset[0]
        opd_gt = batch["opd"]
        pupilGT = batch["pupil"]
        gain = batch["loop_gain"]
        leak = batch["loop_leak"]
        # batch["nphotons"]/batch["ron"] are unused here -- they drive the *simulator's own*
        # noise model (wfs.SetPhotonsAndRON), which never runs in this loop; real camera
        # noise comes from the bench itself.

        z_estimated = torch.zeros(opd_gt.shape[0], M2C.shape[1], device=device)
        z_buffer = torch.zeros_like(z_estimated)
        opd_reconstructed = torch.zeros_like(opd_gt)

    total_loss = 0
    for i in range(CLOSED_LOOP_ITERATIONS):
        # with torch.no_grad():
        if i > 0:
            batch = dataset[i]
            opd_gt = batch["opd"]
            pupilGT = batch["pupil"]

        residual_opd = opd_gt - opd_reconstructed
        z_estimated = z_estimated * leak + gain * z_buffer

        # --- the one line this notebook replaces from Trainer.train() (AI4AO/Trainer.py:101) ---
        wfs_frames = GetBenchWFSFrames(residual_opd)
        # ------------------------------------------------------------------------------------

        preprocessed_frames = framePreprocessor.ProcessFrame(wfs_frames)

        z_output = phaseReconstructor(preprocessed_frames)
        z_buffer = torch.clone(z_output)
        opd_reconstructed = dm(z_estimated @ M2C_T)

        iter_odp = dm(z_output @ M2C_T)
        I_pred = wfs(iter_odp + ref_opd*1e-6, ref_pupil)
        I_pred_norm = I_pred / I_pred.sum(dim=(-2, -1), keepdim=True)
        total_loss = total_loss + torch.mean((I_pred_norm - wfs_frames)** 2) * 1e6 / CLOSED_LOOP_ITERATIONS

    optimizer_bench.zero_grad(set_to_none=True)
    total_loss.backward()
    optimizer_bench.step()

    loss_tracker_bench[u] = total_loss.detach()
    progressBar.set_postfix({"Loss": float(total_loss.detach())})

In [ ]:
imshow_multiple([
    {"tensor": wfs_frames},
    {"tensor": I_pred_norm},
    {"tensor": torch.abs(wfs_frames - I_pred_norm), "scale_reference": wfs_frames}
    ], max_channel_number=9)
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(loss_tracker_bench.cpu().numpy())
ax.set_xlabel("Step")
ax.set_ylabel("Bench physics-consistency loss")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.subplot(121)
imshow(ref_opd*1e3, colorbar=True)
plt.title("Static OPD in nm")
plt.subplot(122)
imshow(ref_pupil, colorbar=True)
plt.title("Static pupil illumination")

In [ ]:
trainer.save_checkpoint(SAVE_CHECKPOINT_PATH, BenchFineTuneSteps=BENCH_FINETUNE_STEPS)
print(f"Saved to {SAVE_CHECKPOINT_PATH}")

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

n_frames = 60
result = trainer.evaluate(n_steps=n_frames, dataset=dataset)

fig, axes = imshow_multiple(
    [
        {"tensor": result.opd[0], "title": "Input opd", "same_scale": True},
        {"tensor": result.residual_opd[0], "title": "Residual opd", "scale_reference": result.opd[0]},
        {"tensor": result.wfs_frames[0], "title": "WFS frame"},
        {"tensor": torch.sqrt(result.psfs[0]), "title": "PSF", "same_scale": True},
    ],
    max_channel_number = 9
)


def update(i):
    imshow_multiple(
        [
            {"tensor": result.opd[i], "title": "Input opd", "same_scale": True},
            {"tensor": result.residual_opd[i], "title": "Residual opd", "scale_reference": result.opd[i]},
            {"tensor": result.wfs_frames[i], "title": "WFS frame"},
            {"tensor": torch.sqrt(result.psfs[i]), "title": "PSF", "same_scale": True},
        ],
        fig=fig, axes=axes, 
        max_channel_number = 9
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())

In [ ]:
flat = opd_to_bench_command(ref_opd*1e-6)
flat = flat.detach().cpu().numpy().astype(np.float32).reshape(-1, 1)
DM_SHM.set_data(-flat)

In [ ]:
DM_SHM.set_data(flat * 0)